Step 1: Install Libraries

In [ ]:
!pip install transformers torch rouge-score nltk


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=8c3ac2c4723ae99c9a557e40e3d8e04709e3ab1c929bdbb20182a14224c19f89
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


Step 2: Load Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('/content/tweets.csv')
df = df[['text']].dropna()

# speed-ku
df = df.sample(200, random_state=42)

df.head()


,text
3495,How many illegal buildings should be demolishe...
5461,Who’s fatality is this tho ????
9794,#OnThisDay 2018 Chinese state media confirmed ...
11105,With any luck you will miss the windstorm on e...
1803,"Inferno on Black Friday 1939: 71 deaths, 3,700..."


Step 3: Text Cleaning

In [ ]:
import re
import nltk
nltk.download('punkt')

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['clean_text'] = df['text'].apply(clean_text)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Step 4: Combine Tweets (Multi-document → Single Summary)

In [ ]:
combined_text = " ".join(df['clean_text'].tolist())


Step 5: Load Transformer Model (BART)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Step 6: Generate Abstractive Summary

In [ ]:
inputs = tokenizer(
    combined_text,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=150,
    min_length=50,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

generated_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("🧠 GENERATED SUMMARY:\n")
print(generated_summary)


🧠 GENERATED SUMMARY:

how many illegal buildings should be demolished in our city in the guadalajara of enrique alfaro. chinese state media confirmed that iranian tanker sanchi had sunk after burning for more than a wee. cumbria county council is refusing to reveal the full cost of a new base for south cumbrias emergency services.


Step 7: ROUGE Evaluation

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

scores = scorer.score(
    combined_text[:1000],  # reference
    generated_summary      # prediction
)

scores


{'rouge1': Score(precision=0.7692307692307693, recall=0.23952095808383234, fmeasure=0.36529680365296807),
 'rouge2': Score(precision=0.6078431372549019, recall=0.18674698795180722, fmeasure=0.2857142857142857),
 'rougeL': Score(precision=0.7307692307692307, recall=0.2275449101796407, fmeasure=0.3470319634703196)}